# Day 2, Session 2 -- Exercises: LangChain Development & Tool Integration

**Engineering context:** You are the engineer building LLM-powered applications with LangChain. Consultants are your users -- they need reusable analysis templates, composable pipelines, custom analytical tools, and knowledge retrieval systems they can query in natural language.

In [2]:
!pip install -q langchain langchain-openai langchain-core langchain-text-splitters python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


## Setup

In [3]:
from dotenv import load_dotenv
load_dotenv()

import json
import os

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("All imports successful!")

All imports successful!


## Recap

In the demos you learned how to use ChatModels and PromptTemplates as LangChain's core building blocks, compose multi-step chains with LCEL's pipe operator, and create custom tools with the `@tool` decorator. You also saw how to split documents into chunks and build a simple RAG pipeline. Now you will build your own set of custom consulting tools and bind them to an LLM.

---
## Task: Custom Consulting Tools

Create three custom tools using the `@tool` decorator that mirror analytical utilities consultants use daily, then bind them to an LLM so it can decide which tool to call. You need to build:

1. **`competitor_analysis_tool(company, competitors)`** -- Takes a focal company (str) and a list of competitors (list[str]). Returns a dict with `focal_company`, `competitors_analyzed` (count), `competitor_list`, `dimensions` (list of analysis dimensions), `methodology`, and a `recommendation` string.

2. **`market_sizing_estimate(total_population, serviceable_pct, obtainable_pct, avg_deal_value)`** -- Calculates TAM (total_population * avg_deal_value), SAM (TAM * serviceable_pct), and SOM (SAM * obtainable_pct). Returns formatted TAM/SAM/SOM values and methodology.

3. **`engagement_roi_tool(engagement_cost, projected_savings, implementation_months)`** -- Calculates ROI percentage and payback period in months. Returns metrics with a verdict: "Strong ROI" if ROI > 200%, "Moderate ROI" if > 100%, otherwise "Marginal ROI".

Then **bind all three tools to an LLM** and test that it can select the right tool for a given query.

Remember: the docstring of each function becomes the tool description that the LLM sees -- write clear, descriptive docstrings.

In [4]:
# Task: Custom Consulting Tools

# TODO 1: Create competitor_analysis_tool using @tool decorator
#   - Parameters: company (str), competitors (list[str])
#   - Return a dict with focal_company, competitors_analyzed, competitor_list,
#     dimensions (e.g., ["Market Share", "Product Portfolio", "Geographic Reach", "Innovation Pipeline"]),
#     methodology, and recommendation

# YOUR CODE HERE


# TODO 2: Create market_sizing_estimate using @tool decorator
#   - Parameters: total_population (float), serviceable_pct (float),
#     obtainable_pct (float), avg_deal_value (float)
#   - Calculate: TAM = total_population * avg_deal_value
#               SAM = TAM * serviceable_pct
#               SOM = SAM * obtainable_pct
#   - Return formatted TAM/SAM/SOM as dollar strings plus methodology

# YOUR CODE HERE


# TODO 3: Create engagement_roi_tool using @tool decorator
#   - Parameters: engagement_cost (float), projected_savings (float),
#     implementation_months (int)
#   - Calculate: roi = ((projected_savings - engagement_cost) / engagement_cost) * 100
#               payback_months = engagement_cost / (projected_savings / 12)
#   - Return dict with engagement_cost, projected_annual_savings, roi_pct,
#     payback_period_months, implementation_timeline, and verdict

# YOUR CODE HERE


# TODO 4: Test each tool directly with .invoke()
# print(competitor_analysis_tool.invoke({"company": "Tesla", "competitors": ["BYD", "Rivian", "Lucid"]}))
# print(market_sizing_estimate.invoke({"total_population": 5_000_000, "serviceable_pct": 0.3, "obtainable_pct": 0.1, "avg_deal_value": 50_000}))
# print(engagement_roi_tool.invoke({"engagement_cost": 2_000_000, "projected_savings": 15_000_000, "implementation_months": 6}))


# TODO 5: Bind all three tools to an LLM and test
# llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
# llm_with_tools = llm.bind_tools([competitor_analysis_tool, market_sizing_estimate, engagement_roi_tool])
# response = llm_with_tools.invoke("Estimate the market size for premium electric SUVs with 10 million potential buyers, 25% serviceable, 8% obtainable, and $65,000 average price")
# print(f"\nLLM tool calls: {response.tool_calls}")

### Expected Output

When you invoke each tool directly and then bind them to an LLM, you should see output similar to the following. The direct tool invocations are deterministic; the LLM tool-call selection may vary in ID but should pick the correct tool and arguments.

In [5]:
expected_output = """
competitor_analysis_tool: {'focal_company': 'Tesla', 'competitors_analyzed': 3,
  'competitor_list': ['BYD', 'Rivian', 'Lucid'],
  'dimensions': ['Market Share', 'Product Portfolio', 'Geographic Reach', 'Innovation Pipeline'],
  'methodology': 'McKinsey Competitive Positioning Framework',
  'recommendation': 'Conduct deep-dive on BYD as primary competitive threat to Tesla'}

market_sizing_estimate: {'TAM': '$250,000,000,000', 'SAM': '$75,000,000,000',
  'SOM': '$7,500,000,000', 'methodology': 'Top-down market sizing with TAM/SAM/SOM framework'}

engagement_roi_tool: {'engagement_cost': '$2,000,000', 'projected_annual_savings': '$15,000,000',
  'roi_pct': '650.0%', 'payback_period_months': '1.6',
  'implementation_timeline': '6 months', 'verdict': 'Strong ROI'}

LLM tool calls: [{'name': 'market_sizing_estimate',
  'args': {'total_population': 10000000, 'serviceable_pct': 0.25,
           'obtainable_pct': 0.08, 'avg_deal_value': 65000},
  'id': '...', 'type': 'tool_call'}]
"""
print(expected_output)


competitor_analysis_tool: {'focal_company': 'Tesla', 'competitors_analyzed': 3,
  'competitor_list': ['BYD', 'Rivian', 'Lucid'],
  'dimensions': ['Market Share', 'Product Portfolio', 'Geographic Reach', 'Innovation Pipeline'],
  'methodology': 'McKinsey Competitive Positioning Framework',
  'recommendation': 'Conduct deep-dive on BYD as primary competitive threat to Tesla'}

market_sizing_estimate: {'TAM': '$250,000,000,000', 'SAM': '$75,000,000,000',
  'SOM': '$7,500,000,000', 'methodology': 'Top-down market sizing with TAM/SAM/SOM framework'}

engagement_roi_tool: {'engagement_cost': '$2,000,000', 'projected_annual_savings': '$15,000,000',
  'roi_pct': '650.0%', 'payback_period_months': '1.6',
  'implementation_timeline': '6 months', 'verdict': 'Strong ROI'}

LLM tool calls: [{'name': 'market_sizing_estimate',
  'args': {'total_population': 10000000, 'serviceable_pct': 0.25,
           'obtainable_pct': 0.08, 'avg_deal_value': 65000},
  'id': '...', 'type': 'tool_call'}]

